In [48]:
import os
import re
import json
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PDF ingestion
from pypdf import PdfReader

# Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# (Optional) local generator
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Distributed computing system. 
from pyspark import SparkContext, SparkConf
from pathlib import Path


In [49]:
# Create a SparkContext
#conf = SparkConf().setAppName("IngestData").setMaster("local[*]")
#sc = SparkContext(conf=conf)
#sc.stop()  # Stop any existing SparkContext (if running in an interactive environment)

# Module 1 — Ingest: extract text from a PDF

In [50]:
@dataclass
class RAGConfig:
    pdf_path: str = "course_material"

    # Chunking (simple word-based chunking for this practical)
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # Retrieval
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "sbert"   # {"tfidf", "sbert"}

    # Embedding models
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Generator (fallback: small, CPU-friendly)
    gen_model_name: str = "google/flan-t5-small"
    max_new_tokens: int = 256

cfg = RAGConfig()

print(cfg)
print("CUDA available:", torch.cuda.is_available())


RAGConfig(pdf_path='course_material', chunk_size_words=200, chunk_overlap_words=50, top_k=4, similarity_metric='cosine', embedding_strategy='sbert', sbert_model_name='sentence-transformers/all-MiniLM-L6-v2', gen_model_name='google/flan-t5-small', max_new_tokens=256)
CUDA available: False


In [51]:
# (Optional) If you're running this on a fresh environment, uncomment and run.
# !pip install -U pypdf numpy pandas scikit-learn sentence-transformers transformers torch tqdm requests

In [52]:
def extract_pages_from_pdf(pdf_path: str, pdf_name: Optional[str] = None) -> List[Dict]:
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : pdf_name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages


def extract_pages_from_folder(folder_path: str) -> List[Dict]:
    """
    Browse a folder and extract pages from all PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.


    Returns
    -------
    results : list of dict
        {
            [
                {"pdf_name": "name_pdf.pdf", "page_num": 1, "text": "..."},
                {"pdf_name": "name_pdf.pdf", "page_num": 2, "text": "..."},
            ]
        }
    """

    folder = Path(folder_path)

    results = list()

    # Get all pdf files in folder path
    pdf_files = list( folder.glob("*.pdf") )

    # For each pdf file
    for pdf_file in tqdm(pdf_files,  desc="Processing PDFs"):
        try:
            # Extract extract the text of each page and save it in a results list
            pages = extract_pages_from_pdf(str(pdf_file), str(pdf_file.name) )

            results += pages

        except Exception as e:
            print(f"Erreur avec {pdf_file.name}: {e}")

    return results

# --- quick sanity check (should not error once implemented) ---
pages = extract_pages_from_folder(cfg.pdf_path)

Processing PDFs:  31%|███       | 9/29 [00:12<00:51,  2.58s/it]Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 127 0 (offset 0)
Processing PDFs:  55%|█████▌    | 16/29 [00:22<00:15,  1.16s/it]Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 79 0 (offset 0)
Ignoring wrong pointing object 96 0 (offset 0)
Ignoring wrong pointing obje

In [53]:
print("Pages:", len(pages))
print("First page preview:\n",f"pdf_name: {pages[0]['pdf_name']}\npage_num: {pages[0]['page_num']}\ntext: {pages[0]['text'][:100]}...")

Pages: 1662
First page preview:
 pdf_name: 1-intro.pdf
page_num: 1
text: INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT
Lecture 1: Introduction, HDFS, Map/Reduce
Dimitris Sacha...
